# Attention-Based Knowledge Retrieval

How attention heads retrieve stored knowledge: query-key matching patterns, value extraction, knowledge routing across heads, retrieval vs computation classification, and factual association strength.

## Why This Matters

Knowledge attention retrieval investigates where and how models store factual information. Locating knowledge in specific components (neurons, layers, directions) is essential for understanding hallucination, enabling fact editing, and assessing what a model truly knows.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — Causal tracing to locate factual knowledge

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_knowledge_retrieval import (
    query_key_matching,
    value_extraction_pattern,
    knowledge_routing,
    retrieval_vs_computation,
    factual_association_strength,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Query-key matching: what does the query at the last position attend to?
result = query_key_matching(model, tokens, layer=0, head=0)
print(f'Query norm: {result["query_norm"]:.4f}')
print(f'Selectivity: {result["selectivity"]:.2f}')
print('Top matches (pos, attn_weight, qk_score):')
for pos, weight, score in result['top_matches']:
    print(f'  Pos {pos}: weight={weight:.4f}, score={score:.4f}')

In [ ]:
# Value extraction: how are values combined?
result = value_extraction_pattern(model, tokens, layer=0, head=0)
print(f'Dominant source: position {result["dominant_source"]}')
print(f'Value diversity: {result["value_diversity"]:.4f}')
print(f'Total output norm: {result["total_output_norm"]:.4f}')
for src in result['per_source'][:3]:
    print(f'  Pos {src["position"]}: weight={src["attention_weight"]:.4f}, norm={src["contribution_norm"]:.4f}')

In [ ]:
# Knowledge routing: which heads contribute most?
result = knowledge_routing(model, tokens)
print(f'Total attention contribution: {result["total_attention_contribution"]:.4f}')
print('Top routing heads:')
for h in result['top_routing_heads']:
    print(f'  L{h["layer"]}H{h["head"]}: norm={h["contribution_norm"]:.4f}')

In [ ]:
# Classify heads as retrieval vs computation
result = retrieval_vs_computation(model, tokens, layer=0)
for h in result['per_head']:
    print(f'Head {h["head"]}: {h["classification"]} '
          f'(retrieval={h["retrieval_score"]:.3f}, '
          f'computation={h["computation_score"]:.3f})')

In [ ]:
# Factual association strength
result = factual_association_strength(model, tokens)
print(f'Aggregate strength: {result["aggregate_strength"]:.4f}')
print('Strongest associations:')
for a in result['strongest_associations']:
    print(f'  L{a["layer"]}H{a["head"]}: val_pos={a["value_position"]}, '
          f'strength={a["association_strength"]:.4f}')